# Attention: from Q/K/V to a causal mask

**Attention Is All You Need** · Vaswani et al. · 2017

[Original paper](https://arxiv.org/abs/1706.03762)

**Learning goal:** Implement scaled dot-product attention without a ready-made attention module and verify that future tokens cannot alter the past.

## Before starting

PyTorch tensors, matrix multiplication, softmax, and backward. Start with “How a model learns”.

Allow roughly 40–60 minutes including exercises. CPU, PyTorch, and Matplotlib; no model or dataset downloads. This is a study-time estimate, not a runtime benchmark.

## Problem and idea

Before Transformers, sequence communication often depended on recurrence. Attention lets each position collect a weighted mixture of other positions. Q represents a query, K matching features, and V the information to transfer; these are learned projections, not fixed linguistic roles.

The output mixes V, with mixing weights determined by Q and K. Multiple heads operate in different subspaces. Attention alone does not encode order: position information is needed. A causal LM must separately prohibit future access.

## Reproduction boundary

Attention mechanism only: no complete encoder–decoder, positional encoding, translation corpus, or paper BLEU reproduction.

The code is an original educational implementation of mathematical ideas from the paper, not a copy of an author repository. Passing assertions verifies the stated implementation properties, not the paper’s results.

## Formula → tensors

`S = Q @ K.transpose(-2,-1) / sqrt(d_head)`; `P = softmax(S + mask)`; `O = P @ V`.

For Q/K/V `[B,H,T,D]`, scores are `[B,H,T,T]`. Softmax runs over keys, the final axis, so each row sums to 1. Scaling controls dot-product variance. An additive mask uses 0 for allowed positions and `-inf` for forbidden positions. Fully masked rows produce NaNs; keep the diagonal available.

In [ ]:
%matplotlib inline
import math
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| CPU | seed=42')

## Step 1

Compute scores and weights explicitly. Above the causal diagonal, probabilities must be zero.

In [ ]:
def attention(q, k, v, causal=True):
    scores = q @ k.transpose(-2, -1) / q.shape[-1]**0.5
    if causal:
        future = torch.triu(torch.ones(q.shape[-2], k.shape[-2], dtype=torch.bool), diagonal=1)
        scores = scores.masked_fill(future, float('-inf'))
    probabilities = scores.softmax(dim=-1)
    return probabilities @ v, probabilities
q, k, v = [torch.randn(2, 2, 6, 4) for _ in range(3)]
out, probabilities = attention(q, k, v)
assert out.shape == q.shape
assert torch.allclose(probabilities.sum(-1), torch.ones(2, 2, 6))
assert torch.equal(probabilities.triu(1), torch.zeros_like(probabilities))
plt.imshow(probabilities[0, 0].detach(), cmap='Reds', vmin=0, vmax=1)
plt.xlabel('key position'); plt.ylabel('query position'); plt.colorbar(); plt.show()

## Step 2

Add learned Q/K/V projections. Gradients should reach all three projections.

In [ ]:
class AttentionBlock(nn.Module):
    def __init__(self, width=8):
        super().__init__()
        self.q = nn.Linear(width, width, bias=False)
        self.k = nn.Linear(width, width, bias=False)
        self.v = nn.Linear(width, width, bias=False)
    def forward(self, x):
        return attention(self.q(x), self.k(x), self.v(x))[0]
block = AttentionBlock(); x = torch.randn(2, 6, 8)
target = torch.randn_like(x)
F.mse_loss(block(x), target).backward()
for name, parameter in block.named_parameters():
    print(name, 'gradient norm:', parameter.grad.norm().item())
    assert torch.isfinite(parameter.grad).all() and parameter.grad.norm() > 0

## Step 3

Change only future inputs. Verify causality through outputs, not class names.

In [ ]:
changed = x.clone(); changed[:, 3:] += 10 * torch.randn_like(changed[:, 3:])
with torch.no_grad():
    delta = (block(x)[:, :3] - block(changed)[:, :3]).abs().max()
    unmasked_a = attention(block.q(x), block.k(x), block.v(x), False)[0]
    unmasked_b = attention(block.q(changed), block.k(changed), block.v(changed), False)[0]
print('causal delta:', delta.item(), 'unmasked delta:', (unmasked_a[:, :3] - unmasked_b[:, :3]).abs().max().item())
assert delta < 1e-6

## Reading the result

Zero future probabilities and unchanged early outputs are separate checks. Nonzero gradients establish a learning path, not language-model quality. This heatmap uses random inputs and has no linguistic interpretation.

## Independent experiments

1. Remove sqrt(D) and compare weight entropy at D=4 and D=256.
2. Remove the mask: which assertion should fail?
3. Add two heads and an output Linear while preserving width 8.

Write a prediction first, then change one factor at a time. Reset the kernel and rerun all cells for a controlled comparison. Save a copy for your changes.

## Reasoning check — after experimenting

For independent normal Q/K, dot-product variance grows with D; unscaled softmax often sharpens. Without masking, early outputs depend on future inputs. Two heads reshape `[B,T,8] → [B,2,T,4]`, followed by concatenation and an 8→8 projection.

## What to record in your lab notes

- Hypothesis and changed parameter.
- Seed, data size, step budget, and precision.
- Before/after diagnostics, plot, and explanation of unexpected results.
- What this experiment **does not** establish.

Use “Open Studio workspace” above the lesson to continue in the related tools.